# Exercise 08 — SOLUTION: Membership Inference Attack

## Background

See student notebook.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(42)
np.random.seed(42)

transform = transforms.ToTensor()
full_train = datasets.MNIST('./data', train=True,  download=True, transform=transform)
full_test  = datasets.MNIST('./data', train=False, download=True, transform=transform)

# 1000-sample training sets: small enough to cause strong overfitting (~10% gap).
# Balanced member/non-member splits so the random-guess baseline is truly 50%.
target_train = Subset(full_train, range(0,    1000))
shadow_train = Subset(full_train, range(1000, 2000))
target_test  = Subset(full_test,  range(0,    1000))
shadow_test  = Subset(full_test,  range(1000, 2000))

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32*7*7,64), nn.ReLU(), nn.Linear(64,10)
        )
    def forward(self, x): return self.net(x)

def train_model(model, dataset, epochs=30):
    loader  = DataLoader(dataset, batch_size=32, shuffle=True)
    opt     = torch.optim.Adam(model.parameters())
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    model.eval()

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=512)
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item(); total += len(y)
    return correct / total

def get_confidence_vectors(model, dataset):
    """Full 10-dim softmax — used for visualization."""
    loader = DataLoader(dataset, batch_size=512)
    confs  = []
    with torch.no_grad():
        for x, _ in loader:
            confs.append(torch.softmax(model(x), dim=1).numpy())
    return np.concatenate(confs)

def get_loss_features(model, dataset):
    """Per-sample [max_confidence, cross_entropy_loss] — the key attack signal.
    Members have loss ≈ 0 (memorized); non-members have loss > 0.
    """
    loader  = DataLoader(dataset, batch_size=512)
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    feats   = []
    with torch.no_grad():
        for x, y in loader:
            logits   = model(x)
            probs    = torch.softmax(logits, dim=1)
            max_conf = probs.max(dim=1).values.numpy()
            loss     = loss_fn(logits, y).numpy()
            feats.append(np.stack([max_conf, loss], axis=1))
    return np.concatenate(feats)

print("Training target model...")
target_model = SmallCNN(); train_model(target_model, target_train)
print(f"  train acc: {evaluate(target_model, target_train):.2%}  test acc: {evaluate(target_model, target_test):.2%}")
print("Training shadow model...")
shadow_model = SmallCNN(); train_model(shadow_model, shadow_train)
print(f"  train acc: {evaluate(shadow_model, shadow_train):.2%}  test acc: {evaluate(shadow_model, shadow_test):.2%}")

shadow_member_feats    = get_loss_features(shadow_model, shadow_train)
shadow_nonmember_feats = get_loss_features(shadow_model, shadow_test)

X_attack = np.concatenate([shadow_member_feats, shadow_nonmember_feats])
y_attack  = np.concatenate([np.ones(len(shadow_member_feats)),
                             np.zeros(len(shadow_nonmember_feats))])
print(f"\nAttack training set: {len(X_attack)} samples")
print(f"  mean loss  — members: {shadow_member_feats[:,1].mean():.4f}  non-members: {shadow_nonmember_feats[:,1].mean():.4f}")

## Attack Model — Solution

In [ ]:
# SOLUTION
attack_model = LogisticRegression(max_iter=1000, random_state=42)
attack_model.fit(X_attack, y_attack)

target_member_feats    = get_loss_features(target_model, target_train)
target_nonmember_feats = get_loss_features(target_model, target_test)
X_target = np.concatenate([target_member_feats, target_nonmember_feats])
y_target  = np.concatenate([np.ones(len(target_member_feats)),
                              np.zeros(len(target_nonmember_feats))])

y_pred = attack_model.predict(X_target)
y_prob = attack_model.predict_proba(X_target)[:, 1]

print(f"Attack accuracy : {accuracy_score(y_target,  y_pred):.3f}  (baseline: 0.500)")
print(f"Attack precision: {precision_score(y_target, y_pred):.3f}")
print(f"Attack recall   : {recall_score(y_target,    y_pred):.3f}")
print(f"Attack AUC      : {roc_auc_score(y_target,   y_prob):.3f}  (baseline: 0.500)")

## Visualize and Discuss

In [ ]:
# Visualize: loss distribution of members vs non-members
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mem_loss  = target_member_feats[:, 1]
nonm_loss = target_nonmember_feats[:, 1]

axes[0].hist(mem_loss,  bins=50, alpha=0.7, label='Members',     color='black')
axes[0].hist(nonm_loss, bins=50, alpha=0.6, label='Non-members', color='salmon')
axes[0].set_xlabel('Per-sample loss'); axes[0].set_ylabel('Count (log scale)')
axes[0].set_yscale('log')
axes[0].set_title('Loss Distribution: Members vs Non-members')
axes[0].legend()

mem_scores  = attack_model.predict_proba(target_member_feats)[:, 1]
nonm_scores = attack_model.predict_proba(target_nonmember_feats)[:, 1]
axes[1].hist(mem_scores,  bins=50, alpha=0.7, label='Members',     color='black')
axes[1].hist(nonm_scores, bins=50, alpha=0.6, label='Non-members', color='salmon')
axes[1].set_xlabel('Attack model membership score'); axes[1].set_ylabel('Count')
axes[1].set_title('Attack Model Score Distribution')
axes[1].legend()

plt.tight_layout(); plt.show()

print("""
Discussion:
- Members have loss ≈ 0 (the model memorized them); non-members have loss > 0.
- This loss gap is the core attack signal — the attack would fail on a well-regularized model.
- The attack works BECAUSE the model overfits. A model with identical train/test loss
  would show no exploitable difference.
- Mitigation: differential privacy (DP-SGD), regularisation, early stopping.
""")